# 3 — Interactive Product Artifact Diagnostics

Use this notebook after a product run. Provide either the product artifact directory or any file inside it.

The notebook reconstructs the **observable agent process** and presents it as one compact interactive workspace:

```text
submitted input
→ product interpretation and uncertainty
→ manufacturer / local / global search route
→ candidate investigation and rejection
→ text, browser and visual evidence
→ strict acceptance rules
→ manufacturer-versus-retailer choice
→ final URL
```

The explorer supports hover detail, panning, zooming, legend isolation, candidate filtering and click-to-zoom evidence hierarchies. It shows recorded evidence, actions, rules, judgments and conclusions—not hidden chain-of-thought.

Detailed usage: `docs/INTERACTIVE_ARTIFACT_DIAGNOSTICS.md`.

In [ ]:
from __future__ import annotations

import importlib
import sys
from pathlib import Path

import pandas as pd

def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / 'docker-compose.yml').is_file() and (candidate / 'src' / 'product_evidence_harness').is_dir():
            return candidate
    raise RuntimeError('Could not locate the web_search_tool repository root')

PROJECT_ROOT = find_project_root()
for required in (str(PROJECT_ROOT), str(PROJECT_ROOT / 'src')):
    if required not in sys.path:
        sys.path.insert(0, required)
importlib.invalidate_caches()

from product_evidence_harness.artifact_diagnostics import (
    build_artifact_diagnostics,
    write_artifact_diagnostic_report,
)
from product_evidence_harness.interactive_artifact_diagnostics import (
    build_interactive_artifact_dashboard,
    display_interactive_artifact_dashboard,
)

print(f'Repository: {PROJECT_ROOT}')
print('Offline diagnostics: the agent service, browser, SerpAPI and LLM are not required.')

## Artifact path

Paste either the artifact directory or any file inside it. The loader resolves the owning product artifact automatically.

In [ ]:
ARTIFACT_PATH = PROJECT_ROOT / 'data' / 'artifacts' / 'ROW-001'
RUN_DIAGNOSTICS = False

print(ARTIFACT_PATH)

In [ ]:
if RUN_DIAGNOSTICS:
    diagnostics = build_artifact_diagnostics(ARTIFACT_PATH)
    dashboard = build_interactive_artifact_dashboard(diagnostics)
    print(f'Loaded artifact: {diagnostics.artifact_dir}')
    print(f'Standalone interactive dashboard: {dashboard.output_path}')
else:
    print('Ready. Set ARTIFACT_PATH and RUN_DIAGNOSTICS = True.')

# Interactive diagnostic workspace

Use the tabs rather than scrolling through large tables:

- **Decision Map** — complete observable process; hover, pan, zoom and isolate stages.
- **Judgment Timeline** — ordered business questions, evidence, rules, rejected alternatives and next actions.
- **Candidates** — filter and compare URLs by feature coverage, confidence and outcome.
- **Evidence** — click through extraction method → source → feature, including visual evidence.
- **Artifacts** — click-to-zoom map of generated files, purpose, path and relative size.

In [ ]:
if 'dashboard' not in globals():
    raise RuntimeError('Run the artifact-loading cell first')

interactive_view = display_interactive_artifact_dashboard(dashboard)

## Shareable outputs

The interactive HTML is the primary visual diagnostic artifact. The Markdown and workbook remain available for audit, export and detailed offline review, but they are not displayed as notebook tables.

In [ ]:
EXPORT_AUXILIARY_REPORTS = True

if EXPORT_AUXILIARY_REPORTS:
    diagnostic_report_path = write_artifact_diagnostic_report(
        diagnostics,
        output_path=diagnostics.artifact_dir / 'artifact_diagnostic_report.md',
    )
    diagnostic_workbook_path = diagnostics.artifact_dir / 'artifact_diagnostic_workbook.xlsx'
    with pd.ExcelWriter(diagnostic_workbook_path, engine='openpyxl') as writer:
        for sheet_name, frame in diagnostics.tables().items():
            frame.to_excel(writer, sheet_name=sheet_name.replace('_df', '')[:31], index=False)

    print(f'Interactive dashboard: {dashboard.output_path}')
    print(f'Diagnostic Markdown: {diagnostic_report_path}')
    print(f'Diagnostic workbook: {diagnostic_workbook_path}')
    print(f"Human judgment artifact: {diagnostics.artifact_dir / 'business_judgement_review.md'}")

## Human review protocol

Open the **Judgment Timeline** first and ask the reviewer to identify the first point where their decision differs. Then use **Candidates** and **Evidence** to inspect exactly what the agent accepted, rejected, missed or overweighted.

The first divergent observable judgment becomes the next product requirement.